In [ ]:
print("Notebook connected") 

Single Droplet Imaging and segmentation. I might want to add some just simple view functions to look at images before I actually start processing but that will be 
added later as I itterate on the code. 

## Updated U-Net training notebook

This version uses `tifffile.memmap`, cluster paths, `.keras` model files, improved droplet/nucleus training labels, and Z-plane focus filtering.

Notes about this Code: 
We will be using the package skimage as our primary image loader and will use it for threashilding and image manipulations. 

In [ ]:
# ============================================================
# Core Imports for U-Net Training Label Development
# ============================================================

from pathlib import Path
import os
import time
import math
import gc

import numpy as np
import pandas as pd

import tifffile as tiff

import matplotlib.pyplot as plt

from skimage import filters, morphology, measure
from skimage.measure import label as cc_label, regionprops_table
from scipy import ndimage as ndi

import tensorflow as tf
from tensorflow.keras import layers, models

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kwargs: x

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Num GPUs Available:", len(tf.config.list_physical_devices("GPU")))

# ============================================================
# Project Directory Setup
# ============================================================

DATA_ROOT = Path("/home/tdeibert/Data/Machine_Learning_Dev/")
MODEL_ROOT = DATA_ROOT / "Models"
IMAGE_ROOT = DATA_ROOT / "Images"
OUT_ROOT = DATA_ROOT / "Outputs"

# Training patch outputs
TRAINING_ROOT = OUT_ROOT / "training_patches_v3"
IMG_OUT = TRAINING_ROOT / "images"
LAB_OUT = TRAINING_ROOT / "labels"
QC_OUT = OUT_ROOT / "training_qc_v3"

for path in [DATA_ROOT, MODEL_ROOT, IMAGE_ROOT, OUT_ROOT, TRAINING_ROOT, IMG_OUT, LAB_OUT, QC_OUT]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "unet_droplet_nucleus_v3"
BEST_MODEL_PATH = MODEL_ROOT / f"{MODEL_NAME}_best.keras"
FINAL_MODEL_PATH = MODEL_ROOT / f"{MODEL_NAME}_final.keras"

print("DATA_ROOT       :", DATA_ROOT)
print("MODEL_ROOT      :", MODEL_ROOT)
print("IMAGE_ROOT      :", IMAGE_ROOT)
print("OUT_ROOT        :", OUT_ROOT)
print("TRAINING_ROOT   :", TRAINING_ROOT)
print("BEST_MODEL_PATH :", BEST_MODEL_PATH)
print("FINAL_MODEL_PATH:", FINAL_MODEL_PATH)

# ============================================================
# Memory-Mapped TIFF Loading
# ============================================================

# Update this filename to the TIFF you want to train from.
# This assumes the file lives in IMAGE_ROOT.
IMAGE_FILE = IMAGE_ROOT / "Untitled.tif"


def load_memmap_tiff(path):
    """
    Load a TIFF as a memory-mapped array.

    This avoids loading the full hyperstack into RAM and should be used
    everywhere in this notebook for large microscopy TIFF files.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Could not find TIFF file: {path}")

    arr = tiff.memmap(str(path))
    print("Loaded memory-mapped TIFF")
    print("  Path :", path)
    print("  Shape:", arr.shape)
    print("  Dtype:", arr.dtype)
    return arr


img_fov = load_memmap_tiff(IMAGE_FILE)
img = img_fov  # backward-compatible alias for older cells


In [ ]:
# ============================================================
# Sanity Check: Paths and Loaded Image
# ============================================================

print("IMAGE_FILE exists:", IMAGE_FILE.exists())
print("Image type:", type(img_fov))
print("Image shape:", img_fov.shape)
print("Image dtype:", img_fov.dtype)


In [ ]:
# ============================================================
# Updated U-Net Training Configuration
# ============================================================

from dataclasses import dataclass
from pathlib import Path


@dataclass
class TrainingConfig:
    # ----------------------------
    # Project paths
    # ----------------------------
    data_root: Path = Path("/home/tdeibert/Data/Machine_Learning_Dev/")
    model_root: Path = data_root / "Models"
    image_root: Path = data_root / "Images"
    out_root: Path = data_root / "Outputs"

    # Uses the memory-mapped TIFF loaded earlier in the notebook.
    image_file: Path = IMAGE_FILE

    # ----------------------------
    # Image metadata
    # Expected shape: (T, Z, C, Y, X)
    # Current example: (10, 20, 3, 3889, 5732)
    # ----------------------------
    n_time: int = 10
    n_z: int = 20
    n_channels: int = 3
    pixel_size_um: float = 0.108

    # ----------------------------
    # Channel order
    # ----------------------------
    membrane_channel_idx: int = 0
    nucleus_channel_idx: int = 1
    npc_channel_idx: int = 2

    # ----------------------------
    # Patch extraction
    # 512 px = 55.3 µm at 0.108 µm/px
    # ----------------------------
    patch_size: int = 512
    patch_jitter_px: int = 128

    nucleus_patch_fraction: float = 0.70
    droplet_patch_fraction: float = 0.20
    hard_negative_patch_fraction: float = 0.10

    patches_per_plane: int = 24
    max_patches_per_epoch: int = 6000

    # ----------------------------
    # Physical object size filters
    # ----------------------------
    min_nucleus_area_um2: float = 50.0
    min_droplet_area_um2: float = 1000.0

    # ----------------------------
    # Nucleus segmentation labels
    # ----------------------------
    nucleus_blur_sigma: float = 1.5
    nucleus_threshold_method: str = "otsu"
    nucleus_closing_radius_um: float = 2.0
    nucleus_hole_area_um2: float = 300.0

    # ----------------------------
    # Droplet segmentation labels
    # ----------------------------
    puncta_clip_quantile: float = 0.985
    droplet_blur_sigma: float = 8.0
    droplet_closing_radius_um: float = 3.0
    droplet_hole_area_um2: float = 8000.0

    # ----------------------------
    # Focus filtering
    # ----------------------------
    use_focus_filter: bool = True
    min_focus_score: float = 0.0005
    exclude_edge_z_planes: bool = True
    edge_z_exclusion: int = 2

    # ----------------------------
    # Model / training
    # ----------------------------
    model_name: str = "unet_droplet_nucleus_v4"
    num_classes: int = 3

    batch_size: int = 2
    epochs: int = 50
    learning_rate: float = 1e-4

    validation_fraction: float = 0.20
    seed: int = 42

    # ----------------------------
    # Derived paths
    # ----------------------------
    @property
    def best_model_path(self):
        return self.model_root / f"{self.model_name}_best.keras"

    @property
    def final_model_path(self):
        return self.model_root / f"{self.model_name}_final.keras"

    @property
    def training_root(self):
        return self.out_root / "training_patches_v4"

    @property
    def image_patch_dir(self):
        return self.training_root / "images"

    @property
    def label_patch_dir(self):
        return self.training_root / "labels"

    @property
    def label_qc_dir(self):
        return self.out_root / "label_qc_v4"


cfg = TrainingConfig()

for p in [
    cfg.data_root,
    cfg.model_root,
    cfg.image_root,
    cfg.out_root,
    cfg.training_root,
    cfg.image_patch_dir,
    cfg.label_patch_dir,
    cfg.label_qc_dir,
]:
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Backward-compatible aliases for older notebook cells
# ------------------------------------------------------------------
DATA_ROOT = cfg.data_root
MODEL_ROOT = cfg.model_root
IMAGE_ROOT = cfg.image_root
OUT_ROOT = cfg.out_root
TRAINING_ROOT = cfg.training_root
IMG_OUT = cfg.image_patch_dir
LAB_OUT = cfg.label_patch_dir
QC_OUT = cfg.label_qc_dir

MODEL_NAME = cfg.model_name
BEST_MODEL_PATH = cfg.best_model_path
FINAL_MODEL_PATH = cfg.final_model_path

MEM_CH = cfg.membrane_channel_idx
NUC_CH = cfg.nucleus_channel_idx
NPC_CH = cfg.npc_channel_idx

PIXEL_SIZE_UM = cfg.pixel_size_um
PATCH_SIZE = cfg.patch_size
PATCH_JITTER_PX = cfg.patch_jitter_px

# Kept for older patch-generation cells that still use stride/min foreground fraction.
STRIDE = cfg.patch_size
MIN_FG_FRAC = 0.01

USE_FOCUS_FILTER = cfg.use_focus_filter
MIN_FOCUS_SCORE = cfg.min_focus_score
EXCLUDE_EDGE_Z = cfg.edge_z_exclusion

print("Training configuration loaded")
print("Image file:", cfg.image_file)
print("Image shape:", img_fov.shape)
print("Expected order: (T, Z, C, Y, X)")
print("Channels: MEM=", MEM_CH, "NUC=", NUC_CH, "NPC=", NPC_CH)
print("Pixel size:", PIXEL_SIZE_UM, "µm/pixel")
print("Patch size:", PATCH_SIZE, "px")
print("Patch field of view:", round(PATCH_SIZE * PIXEL_SIZE_UM, 2), "µm")
print("Min nucleus area:", cfg.min_nucleus_area_um2, "µm² ->", int(np.ceil(cfg.min_nucleus_area_um2 / (PIXEL_SIZE_UM ** 2))), "px")
print("Min droplet area:", cfg.min_droplet_area_um2, "µm² ->", int(np.ceil(cfg.min_droplet_area_um2 / (PIXEL_SIZE_UM ** 2))), "px")
print("Best model:", BEST_MODEL_PATH)
print("Final model:", FINAL_MODEL_PATH)


## Configuration update notes

This notebook now uses the corrected pixel size (`0.108 µm/pixel`), the current channel order `(membrane, nucleus, NPC)`, a `512 × 512` patch size, and native `.keras` model saving. The older variable names are still defined as aliases so downstream cells continue to run while we refactor the training pipeline.

In [ ]:
#### Simple Viewer Function for Single Droplet Image#### 
import numpy as np
import matplotlib.pyplot as plt
from skimage import filters, morphology

# choose a time and z to play with
t_idx = 9  # 0..9 set the time point that you wish to view. In python 0 is the first indexing number. So time 1 is actually t index 0. 
z_idx = 16  # 0..19 set the z slice that you wish to view. In python 0 is the first indexing number. So z slice 1 is actually z index 0.

NUC_CH = 1   # <-- Nuclus Channel When using a multi channel hyper stack you will define which of your channels contains which feature. 
MEM_CH = 2   # <-- membrane / NPC channel

nuc2d = img[t_idx, z_idx, :, :, NUC_CH] # Extract 2D slice for nuclear channel at specified time and z index
mem2d = img[t_idx, z_idx, :, :, MEM_CH] # Extract 2D slice for membrane/NPC channel at specified time and z index

print("nuc2d shape:", nuc2d.shape) #desctiptors of the extracted image
print("mem2d shape:", mem2d.shape) #descriptors of the extracted image

# Display the images 
fig, ax = plt.subplots(1, 2, figsize=(8, 4)) # plt.subplots comes from matplotlib and is used to create a figure and a set of subplots. Fig, defindes the entire figure. where has 1, 2 is the differnt channels which we define below. Think of this as setting up the container for the plot. 
ax[0].imshow(nuc2d, cmap="gray"); ax[0].set_title("Nuclear channel"); ax[0].axis("off") # have defined 0 to use nuc2d image with index from above and given it a title. 
ax[1].imshow(mem2d, cmap="gray"); ax[1].set_title("NPC channel"); ax[1].axis("off") # we have defined 1 to use mem2d image with index from above and given it a title.
plt.tight_layout(); plt.show() #this is the actual command that displays the plot. plt. is the function where as tight_layout() is the format. 


In the following section we will define our threasholding for the various parts of the image for generating training images. 
In this example we are identifying three objects for image segmentation. Droplets, background, and Nuclei

In [ ]:
# ============================================================
# Training Label Segmentation Utilities v3
# ============================================================


def area_um2_to_px(area_um2, pixel_size_um=PIXEL_SIZE_UM):
    """Convert area in µm² to pixels."""
    return int(np.ceil(area_um2 / (pixel_size_um ** 2)))


def radius_um_to_px(radius_um, pixel_size_um=PIXEL_SIZE_UM):
    """Convert radius in µm to pixels."""
    return max(1, int(np.ceil(radius_um / pixel_size_um)))


def normalize_01(img2d):
    """Normalize a 2D image to [0, 1] safely."""
    arr = img2d.astype("float32", copy=False)
    lo = np.nanmin(arr)
    hi = np.nanmax(arr)
    if hi <= lo:
        return np.zeros_like(arr, dtype="float32")
    return (arr - lo) / (hi - lo)


def normalize_channel(ch2d):
    """Backward-compatible channel normalizer for model input building."""
    return normalize_01(ch2d).astype("float32")


def focus_score_variance_laplacian(img2d):
    """
    Focus score based on variance of the Laplacian.

    Higher = sharper/more in-focus.
    Lower = blurrier/out-of-focus.
    """
    img_f = normalize_01(img2d)
    lap = ndi.laplace(img_f)
    return float(np.var(lap))


def z_plane_allowed(z, n_z, exclude_edge_z=EXCLUDE_EDGE_Z):
    """Exclude the highest and lowest Z planes where artifacts are common."""
    return exclude_edge_z <= z < (n_z - exclude_edge_z)


def plane_passes_focus(nuc2d, min_focus_score=MIN_FOCUS_SCORE):
    """Return boolean pass/fail plus focus score."""
    score = focus_score_variance_laplacian(nuc2d)
    return score >= min_focus_score, score


def segment_droplet_from_npc(
    npc2d,
    pixel_size_um=PIXEL_SIZE_UM,
    puncta_clip_quantile=0.985,
    blur_sigma=8.0,
    min_droplet_area_um2=500,
    hole_area_um2=5000,
    closing_radius_um=3.0,
    return_debug=False,
):
    """
    Segment droplet-scale signal from the NPC channel while suppressing NPC puncta.

    Important behavior:
    - The brightest NPC puncta are clipped by percentile.
    - The clipped image is heavily blurred so droplet-scale low-frequency signal remains.
    - The resulting label should teach the model the droplet region, not the NPC puncta.
    """
    npc_f = normalize_01(npc2d)

    clip_value = np.quantile(npc_f, puncta_clip_quantile)
    npc_clean = npc_f.copy()
    npc_clean[npc_clean > clip_value] = clip_value
    npc_clean = normalize_01(npc_clean)

    npc_blur = filters.gaussian(
        npc_clean,
        sigma=blur_sigma,
        preserve_range=True,
    )

    threshold = filters.threshold_otsu(npc_blur)
    droplet_mask = npc_blur > threshold

    droplet_mask = morphology.binary_closing(
        droplet_mask,
        morphology.disk(radius_um_to_px(closing_radius_um, pixel_size_um)),
    )

    droplet_mask = morphology.remove_small_objects(
        droplet_mask,
        min_size=area_um2_to_px(min_droplet_area_um2, pixel_size_um),
    )

    droplet_mask = morphology.remove_small_holes(
        droplet_mask,
        area_threshold=area_um2_to_px(hole_area_um2, pixel_size_um),
    )

    droplet_mask = droplet_mask.astype(bool)

    if return_debug:
        return droplet_mask, {
            "npc_norm": npc_f,
            "npc_clean": npc_clean,
            "npc_blur": npc_blur,
            "clip_value": float(clip_value),
            "threshold": float(threshold),
        }

    return droplet_mask


def segment_nucleus_from_import(
    nuc2d,
    pixel_size_um=PIXEL_SIZE_UM,
    min_nucleus_area_um2=50,
    max_nucleus_area_um2=None,
    blur_sigma=1.5,
    threshold_method="otsu",
    closing_radius_um=1.5,
    hole_area_um2=200,
    return_debug=False,
):
    """
    Segment nuclei as filled solid objects.

    This intentionally closes fragmented/rim labels and fills holes so the model
    learns nucleus interiors, not just nuclear edges.
    """
    nuc_f = normalize_01(nuc2d)
    nuc_blur = filters.gaussian(nuc_f, sigma=blur_sigma, preserve_range=True)

    if threshold_method == "otsu":
        threshold = filters.threshold_otsu(nuc_blur)
    elif threshold_method == "yen":
        threshold = filters.threshold_yen(nuc_blur)
    elif threshold_method == "li":
        threshold = filters.threshold_li(nuc_blur)
    else:
        raise ValueError("threshold_method must be 'otsu', 'yen', or 'li'.")

    nucleus_mask = nuc_blur > threshold

    nucleus_mask = morphology.binary_closing(
        nucleus_mask,
        morphology.disk(radius_um_to_px(closing_radius_um, pixel_size_um)),
    )

    nucleus_mask = ndi.binary_fill_holes(nucleus_mask)

    nucleus_mask = morphology.remove_small_holes(
        nucleus_mask,
        area_threshold=area_um2_to_px(hole_area_um2, pixel_size_um),
    )

    nucleus_mask = morphology.remove_small_objects(
        nucleus_mask,
        min_size=area_um2_to_px(min_nucleus_area_um2, pixel_size_um),
    )

    if max_nucleus_area_um2 is not None:
        max_px = area_um2_to_px(max_nucleus_area_um2, pixel_size_um)
        labeled = measure.label(nucleus_mask)
        cleaned = np.zeros_like(nucleus_mask, dtype=bool)
        for region in measure.regionprops(labeled):
            if region.area <= max_px:
                cleaned[labeled == region.label] = True
        nucleus_mask = cleaned

    nucleus_mask = nucleus_mask.astype(bool)

    if return_debug:
        return nucleus_mask, {
            "nuc_norm": nuc_f,
            "nuc_blur": nuc_blur,
            "threshold": float(threshold),
        }

    return nucleus_mask


def make_3class_label_full(droplet_mask, nucleus_mask):
    """
    3-class training label.

    0 = image background
    1 = droplet interior
    2 = nucleus

    Nucleus overrides droplet.
    """
    label = np.zeros(droplet_mask.shape, dtype=np.uint8)
    label[droplet_mask] = 1
    label[nucleus_mask] = 2
    return label


def build_training_label_plane(
    npc2d,
    nuc2d,
    pixel_size_um=PIXEL_SIZE_UM,
    min_focus_score=MIN_FOCUS_SCORE,
    use_focus_filter=USE_FOCUS_FILTER,
    droplet_kwargs=None,
    nucleus_kwargs=None,
):
    """Build one 3-class training label plane with focus metadata."""
    droplet_kwargs = droplet_kwargs or {}
    nucleus_kwargs = nucleus_kwargs or {}

    focus_pass, focus_score = plane_passes_focus(nuc2d, min_focus_score=min_focus_score)

    if use_focus_filter and not focus_pass:
        return None, {
            "focus_pass": False,
            "focus_score": focus_score,
            "droplet_area_px": 0,
            "nucleus_area_px": 0,
            "reason": "out_of_focus",
        }, None

    droplet_mask, droplet_debug = segment_droplet_from_npc(
        npc2d,
        pixel_size_um=pixel_size_um,
        return_debug=True,
        **droplet_kwargs,
    )

    nucleus_mask, nucleus_debug = segment_nucleus_from_import(
        nuc2d,
        pixel_size_um=pixel_size_um,
        return_debug=True,
        **nucleus_kwargs,
    )

    label = make_3class_label_full(droplet_mask, nucleus_mask)

    metadata = {
        "focus_pass": True,
        "focus_score": focus_score,
        "droplet_area_px": int(droplet_mask.sum()),
        "nucleus_area_px": int(nucleus_mask.sum()),
        "reason": "included",
    }

    debug = {
        **droplet_debug,
        **nucleus_debug,
        "droplet_mask": droplet_mask,
        "nucleus_mask": nucleus_mask,
    }

    return label, metadata, debug


## Training label QC

The next cells let you tune droplet puncta suppression, filled nucleus labels, and focus filtering before building patches.

Segmentation of the nuclear signal. we use the same functions as above. See above section for detailed explainations of line functions. 

In [ ]:
#### Nucleus Segmentation Based on Gaussian Blur and Otsu Thresholding ####
from skimage import filters, morphology
def segment_nucleus_from_import(nuc2d, sigma=1.0, min_size=50):
    nuc_f = nuc2d.astype("float32")
    nuc_f = (nuc_f - nuc_f.min()) / (nuc_f.max() - nuc_f.min() + 1e-8)

    blur = filters.gaussian(nuc_f, sigma=sigma)
    thr = filters.threshold_otsu(blur)
    nucleus_mask = blur > thr

    nucleus_mask = morphology.remove_small_objects(nucleus_mask,
                                                   min_size=min_size)
    nucleus_mask = morphology.remove_small_holes(nucleus_mask,
                                                 area_threshold=min_size)
    return nucleus_mask


Now we will use the masks we generated above to define our objects of interest for our training set. 

In [ ]:
#### Image Labeling to Select Droplet Nearest Nucleus for Machine Learning ####
from skimage.measure import label as cc_label, regionprops

def select_droplet_near_nucleus(droplet_mask, nucleus_mask): #droplet_mask and nucleus_mask were defined above when we created the binary masks for the droplet and nucleus.
    """
    droplet_mask: binary mask with possibly multiple droplets
    nucleus_mask: binary mask for nucleus
    Returns: binary mask of the single droplet associated with that nucleus
    """
    lab = cc_label(droplet_mask)  # using cc_label here we are defining lab as the connected components label of the droplet mask. This will label each connected component (droplet) with a unique integer.
    props = regionprops(lab) # regionprops is a function from skimage that calculates properties of labeled image regions.

    if not props:
        return np.zeros_like(droplet_mask, dtype=bool)

    # nucleus centroid
    ny, nx = np.argwhere(nucleus_mask).mean(axis=0) #np.argwhere is a numpy function that returns the indices of the elements that are non-zero (True) in the input array. Here we are using it to find the centroid of the nucleus.

    # find droplet whose centroid is closest to nucleus centroid
    best_label = None # None is a placeholder for the best label until the loop finds the closest droplet.
    best_dist = np.inf #np.inf represents infinity in numpy. It is used here to initialize the best distance variable to a very large value. Ensuring that the droplet is the best canidate. 
    for p in props: #this is how we intiate a loop in python. We are looping over each droplet region found in the droplet mask. p is being defined in the for loop we don't define. 
        cy, cx = p.centroid #returns row, column or y,x coordinates of the centroid of the droplet region p.
        dist = (cy - ny)**2 + (cx - nx)**2 #calculates the squared distance between the centroid of the droplet region and the centroid of the nucleus.
        if dist < best_dist: # if the calculated distance is less than the best distance found so far we update the best distance and best label.
            best_dist = dist 
            best_label = p.label

    return lab == best_label 



In [ ]:
##### Create 3-Class Label Image With Droplets, BG, And Nuclei Classified ####
def make_3class_label(droplet_mask, nucleus_mask):
    nucleus_mask = nucleus_mask & droplet_mask  # enforce inside droplet
    label = np.zeros_like(droplet_mask, dtype=np.uint8)
    label[droplet_mask] = 1
    label[nucleus_mask] = 2
    return label


In [ ]:
# ============================================================
# QC: Build and Visualize One Training Plane
# ============================================================

# Choose a representative in-focus Z plane.
t = 0
z = min(10, img_fov.shape[1] - 1)

npc2d = img_fov[t, z, :, :, NPC_CH]
nuc2d = img_fov[t, z, :, :, NUC_CH]

label_test, metadata_test, debug_test = build_training_label_plane(
    npc2d=npc2d,
    nuc2d=nuc2d,
    pixel_size_um=PIXEL_SIZE_UM,
    droplet_kwargs={
        "puncta_clip_quantile": 0.985,
        "blur_sigma": 8.0,
        "min_droplet_area_um2": 500,
        "hole_area_um2": 5000,
        "closing_radius_um": 3.0,
    },
    nucleus_kwargs={
        "min_nucleus_area_um2": 50,
        "blur_sigma": 1.5,
        "threshold_method": "otsu",
        "closing_radius_um": 1.5,
        "hole_area_um2": 200,
    },
)

print(metadata_test)

if label_test is not None:
    fig, ax = plt.subplots(2, 3, figsize=(15, 9))

    ax[0, 0].imshow(npc2d, cmap="gray")
    ax[0, 0].set_title("Raw NPC channel")

    ax[0, 1].imshow(debug_test["npc_blur"], cmap="gray")
    ax[0, 1].set_title("NPC clipped + low-pass blur")

    ax[0, 2].imshow(debug_test["droplet_mask"], cmap="gray")
    ax[0, 2].set_title("Droplet training mask")

    ax[1, 0].imshow(nuc2d, cmap="gray")
    ax[1, 0].set_title("Raw nucleus channel")

    ax[1, 1].imshow(debug_test["nucleus_mask"], cmap="gray")
    ax[1, 1].set_title("Filled nucleus training mask")

    ax[1, 2].imshow(label_test, cmap="viridis", vmin=0, vmax=2)
    ax[1, 2].set_title("Final label: 0 bg, 1 droplet, 2 nucleus")

    for axis in ax.ravel():
        axis.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Plane excluded:", metadata_test)


Now we are applying what we tested using a single droplet above into a large FOV image. I have identified a fatal flaw that I must correct though. I have function defintions from the above section baked into this section. So if I don't run above this wont work. 


In [ ]:
# ============================================================
# Large FOV Image Alias
# ============================================================

# The large field-of-view image is already loaded above as a memory-mapped TIFF.
# Keeping this cell for compatibility with older notebook sections.
img_fov = img

print("FOV Loaded as memory map:")
print("  Type :", type(img_fov))
print("  Shape:", img_fov.shape)
print("  Dtype:", img_fov.dtype)


In [ ]:
#### Visualization of Lage FOV Image Channels ####
#Currently clunky but works for now. I might add channel selection etc later
# Need to remember that it defines T1 as 0. So for a 10 timepoint image, T10 is index 9. 
import matplotlib.pyplot as plt

t = 9
z = 10

fig, ax = plt.subplots(1, 3, figsize=(18, 6))

for c in range(3):
    ax[c].imshow(img_fov[t, z, :, :, c], cmap='gray')
    ax[c].set_title(f"Channel {c}")
    ax[c].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
#### Threasholding Segment for Multi-Droplet Images #### 
import pandas as pd

def analyze_field(npc2d, nuc2d):
    # 1) segment
    droplet_mask = segment_droplet_from_npc(npc2d)
    nucleus_mask = segment_nucleus_from_import(nuc2d)

    # 2) label connected components
    droplet_labels = cc_label(droplet_mask)  # 0 = background, 1..Ndroplets
    nucleus_labels = cc_label(nucleus_mask)  # 0 = background, 1..Nnuclei

    # 3) measure droplets and nuclei
    droplet_props = regionprops_table(
        droplet_labels,
        properties=["label", "area", "centroid"]
    )
    nuc_props = regionprops_table(
        nucleus_labels,
        properties=["label", "area", "centroid"]
    )

    droplet_df = pd.DataFrame(droplet_props).rename(
        columns={"label": "droplet_label",
                 "centroid-0": "droplet_cy",
                 "centroid-1": "droplet_cx"}
    )
    nuc_df = pd.DataFrame(nuc_props).rename(
        columns={"label": "nucleus_label",
                 "centroid-0": "nuc_cy",
                 "centroid-1": "nuc_cx"}
    )

    # 4) for each nucleus, find which droplet it belongs to
    droplet_ids_for_nuclei = []
    for cy, cx in zip(nuc_df["nuc_cy"], nuc_df["nuc_cx"]):
        iy, ix = int(round(cy)), int(round(cx))
        droplet_id = droplet_labels[iy, ix]  # 0 if not in any droplet
        droplet_ids_for_nuclei.append(droplet_id)

    nuc_df["droplet_label"] = droplet_ids_for_nuclei

    # keep only nuclei that lie inside a droplet
    nuc_df = nuc_df[nuc_df["droplet_label"] > 0].reset_index(drop=True)

    # 5) optional: merge info (e.g., droplet size with its nuclei)
    merged = nuc_df.merge(droplet_df, on="droplet_label", how="left",
                          suffixes=("_nuc", "_drop"))

    return droplet_labels, nucleus_labels, droplet_df, nuc_df, merged


In [ ]:
def make_3class_label_full(droplet_labels, nucleus_labels):
    label_3c = np.zeros_like(droplet_labels, dtype=np.uint8)
    label_3c[droplet_labels > 0] = 1
    label_3c[nucleus_labels > 0] = 2  # overwrite where nuclei are
    return label_3c


In [ ]:
#### Just a sanity check to extract channels from large FOV image ####
t = 9
z = 15

# Extract the correct channels from your 5D hyperstack
npc2d = img_fov[t, z, :, :, NPC_CH]   # NPC channel (droplet channel)
nuc2d = img_fov[t, z, :, :, NUC_CH]   # nuclear-import channel (nucleus channel)

print("npc2d shape:", npc2d.shape)
print("nuc2d shape:", nuc2d.shape)


In [ ]:
from skimage.measure import label as cc_label, regionprops, regionprops_table

droplet_labels, nucleus_labels, droplet_df, nuc_df, merged = analyze_field(npc2d, nuc2d)
label_3c = make_3class_label_full(droplet_labels, nucleus_labels)


In [ ]:
print("droplet_labels shape:", droplet_labels.shape)
print("nucleus_labels shape:", nucleus_labels.shape)
print("unique droplet labels:", np.unique(droplet_labels)[:10])
print("unique nucleus labels:", np.unique(nucleus_labels)[:10])


In [ ]:
#### Visualization of Segmentation Results on Large FOV Image ####
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: raw intensity images
ax[0, 0].imshow(npc2d, cmap='gray')
ax[0, 0].set_title("NPC channel (droplet source)")
ax[0, 0].axis('off')

ax[0, 1].imshow(nuc2d, cmap='gray')
ax[0, 1].set_title("Nuclear-import channel")
ax[0, 1].axis('off')

ax[0, 2].imshow(label_3c, cmap='tab20')
ax[0, 2].set_title("3-class label (0 bg, 1 droplet, 2 nucleus)")
ax[0, 2].axis('off')

# Row 2: overlays
ax[1, 0].imshow(npc2d, cmap='gray')
ax[1, 0].contour(droplet_labels > 0, colors='red', linewidths=0.5)
ax[1, 0].set_title("Droplet contours (red) on NPC")
ax[1, 0].axis('off')

ax[1, 1].imshow(nuc2d, cmap='gray')
ax[1, 1].contour(nucleus_labels > 0, colors='cyan', linewidths=0.5)
ax[1, 1].set_title("Nucleus contours (cyan) on import")
ax[1, 1].axis('off')

ax[1, 2].imshow(npc2d, cmap='gray')
ax[1, 2].contour(droplet_labels > 0, colors='red', linewidths=0.5)
ax[1, 2].contour(nucleus_labels > 0, colors='cyan', linewidths=0.5)
ax[1, 2].set_title("Droplets (red) + nuclei (cyan)")
ax[1, 2].axis('off')

plt.tight_layout()
plt.show()


This is where the start of Building our Training Data Set for machine learning Starts Currenrly has some dependancies upstream that will need to be fixed. 

In [ ]:
# ============================================================
# Image / Training Configuration
# ============================================================

# Expected hyperstack shape used by this notebook:
#     (T, Z, Y, X, C)
# If your file is not in this order, transpose it before training.
print("Hyperstack shape:", img_fov.shape)

MEM_CH = 0      # membrane/context channel
NUC_CH = 1      # nuclear-import / nucleus-interior channel
NPC_CH = 2      # NPC channel used to infer droplet-scale signal

PIXEL_SIZE_UM = 0.612  # verify for this acquisition

PATCH_SIZE = 256
STRIDE = 256
MIN_FG_FRAC = 0.01

# Z handling
EXCLUDE_EDGE_Z = 2             # excludes first/last N z-slices before focus scoring
MIN_FOCUS_SCORE = 0.0005       # tune after plotting focus scores
USE_FOCUS_FILTER = True

print(f"Channels: MEM={MEM_CH}, NUC={NUC_CH}, NPC={NPC_CH}")
print(f"Pixel size: {PIXEL_SIZE_UM} µm / pixel")


In [ ]:
# ============================================================
# Training Label Segmentation Utilities v3
# ============================================================


def area_um2_to_px(area_um2, pixel_size_um=PIXEL_SIZE_UM):
    """Convert area in µm² to pixels."""
    return int(np.ceil(area_um2 / (pixel_size_um ** 2)))


def radius_um_to_px(radius_um, pixel_size_um=PIXEL_SIZE_UM):
    """Convert radius in µm to pixels."""
    return max(1, int(np.ceil(radius_um / pixel_size_um)))


def normalize_01(img2d):
    """Normalize a 2D image to [0, 1] safely."""
    arr = img2d.astype("float32", copy=False)
    lo = np.nanmin(arr)
    hi = np.nanmax(arr)
    if hi <= lo:
        return np.zeros_like(arr, dtype="float32")
    return (arr - lo) / (hi - lo)


def normalize_channel(ch2d):
    """Backward-compatible channel normalizer for model input building."""
    return normalize_01(ch2d).astype("float32")


def focus_score_variance_laplacian(img2d):
    """
    Focus score based on variance of the Laplacian.

    Higher = sharper/more in-focus.
    Lower = blurrier/out-of-focus.
    """
    img_f = normalize_01(img2d)
    lap = ndi.laplace(img_f)
    return float(np.var(lap))


def z_plane_allowed(z, n_z, exclude_edge_z=EXCLUDE_EDGE_Z):
    """Exclude the highest and lowest Z planes where artifacts are common."""
    return exclude_edge_z <= z < (n_z - exclude_edge_z)


def plane_passes_focus(nuc2d, min_focus_score=MIN_FOCUS_SCORE):
    """Return boolean pass/fail plus focus score."""
    score = focus_score_variance_laplacian(nuc2d)
    return score >= min_focus_score, score


def segment_droplet_from_npc(
    npc2d,
    pixel_size_um=PIXEL_SIZE_UM,
    puncta_clip_quantile=0.985,
    blur_sigma=8.0,
    min_droplet_area_um2=500,
    hole_area_um2=5000,
    closing_radius_um=3.0,
    return_debug=False,
):
    """
    Segment droplet-scale signal from the NPC channel while suppressing NPC puncta.

    Important behavior:
    - The brightest NPC puncta are clipped by percentile.
    - The clipped image is heavily blurred so droplet-scale low-frequency signal remains.
    - The resulting label should teach the model the droplet region, not the NPC puncta.
    """
    npc_f = normalize_01(npc2d)

    clip_value = np.quantile(npc_f, puncta_clip_quantile)
    npc_clean = npc_f.copy()
    npc_clean[npc_clean > clip_value] = clip_value
    npc_clean = normalize_01(npc_clean)

    npc_blur = filters.gaussian(
        npc_clean,
        sigma=blur_sigma,
        preserve_range=True,
    )

    threshold = filters.threshold_otsu(npc_blur)
    droplet_mask = npc_blur > threshold

    droplet_mask = morphology.binary_closing(
        droplet_mask,
        morphology.disk(radius_um_to_px(closing_radius_um, pixel_size_um)),
    )

    droplet_mask = morphology.remove_small_objects(
        droplet_mask,
        min_size=area_um2_to_px(min_droplet_area_um2, pixel_size_um),
    )

    droplet_mask = morphology.remove_small_holes(
        droplet_mask,
        area_threshold=area_um2_to_px(hole_area_um2, pixel_size_um),
    )

    droplet_mask = droplet_mask.astype(bool)

    if return_debug:
        return droplet_mask, {
            "npc_norm": npc_f,
            "npc_clean": npc_clean,
            "npc_blur": npc_blur,
            "clip_value": float(clip_value),
            "threshold": float(threshold),
        }

    return droplet_mask


def segment_nucleus_from_import(
    nuc2d,
    pixel_size_um=PIXEL_SIZE_UM,
    min_nucleus_area_um2=50,
    max_nucleus_area_um2=None,
    blur_sigma=1.5,
    threshold_method="otsu",
    closing_radius_um=1.5,
    hole_area_um2=200,
    return_debug=False,
):
    """
    Segment nuclei as filled solid objects.

    This intentionally closes fragmented/rim labels and fills holes so the model
    learns nucleus interiors, not just nuclear edges.
    """
    nuc_f = normalize_01(nuc2d)
    nuc_blur = filters.gaussian(nuc_f, sigma=blur_sigma, preserve_range=True)

    if threshold_method == "otsu":
        threshold = filters.threshold_otsu(nuc_blur)
    elif threshold_method == "yen":
        threshold = filters.threshold_yen(nuc_blur)
    elif threshold_method == "li":
        threshold = filters.threshold_li(nuc_blur)
    else:
        raise ValueError("threshold_method must be 'otsu', 'yen', or 'li'.")

    nucleus_mask = nuc_blur > threshold

    nucleus_mask = morphology.binary_closing(
        nucleus_mask,
        morphology.disk(radius_um_to_px(closing_radius_um, pixel_size_um)),
    )

    nucleus_mask = ndi.binary_fill_holes(nucleus_mask)

    nucleus_mask = morphology.remove_small_holes(
        nucleus_mask,
        area_threshold=area_um2_to_px(hole_area_um2, pixel_size_um),
    )

    nucleus_mask = morphology.remove_small_objects(
        nucleus_mask,
        min_size=area_um2_to_px(min_nucleus_area_um2, pixel_size_um),
    )

    if max_nucleus_area_um2 is not None:
        max_px = area_um2_to_px(max_nucleus_area_um2, pixel_size_um)
        labeled = measure.label(nucleus_mask)
        cleaned = np.zeros_like(nucleus_mask, dtype=bool)
        for region in measure.regionprops(labeled):
            if region.area <= max_px:
                cleaned[labeled == region.label] = True
        nucleus_mask = cleaned

    nucleus_mask = nucleus_mask.astype(bool)

    if return_debug:
        return nucleus_mask, {
            "nuc_norm": nuc_f,
            "nuc_blur": nuc_blur,
            "threshold": float(threshold),
        }

    return nucleus_mask


def make_3class_label_full(droplet_mask, nucleus_mask):
    """
    3-class training label.

    0 = image background
    1 = droplet interior
    2 = nucleus

    Nucleus overrides droplet.
    """
    label = np.zeros(droplet_mask.shape, dtype=np.uint8)
    label[droplet_mask] = 1
    label[nucleus_mask] = 2
    return label


def build_training_label_plane(
    npc2d,
    nuc2d,
    pixel_size_um=PIXEL_SIZE_UM,
    min_focus_score=MIN_FOCUS_SCORE,
    use_focus_filter=USE_FOCUS_FILTER,
    droplet_kwargs=None,
    nucleus_kwargs=None,
):
    """Build one 3-class training label plane with focus metadata."""
    droplet_kwargs = droplet_kwargs or {}
    nucleus_kwargs = nucleus_kwargs or {}

    focus_pass, focus_score = plane_passes_focus(nuc2d, min_focus_score=min_focus_score)

    if use_focus_filter and not focus_pass:
        return None, {
            "focus_pass": False,
            "focus_score": focus_score,
            "droplet_area_px": 0,
            "nucleus_area_px": 0,
            "reason": "out_of_focus",
        }, None

    droplet_mask, droplet_debug = segment_droplet_from_npc(
        npc2d,
        pixel_size_um=pixel_size_um,
        return_debug=True,
        **droplet_kwargs,
    )

    nucleus_mask, nucleus_debug = segment_nucleus_from_import(
        nuc2d,
        pixel_size_um=pixel_size_um,
        return_debug=True,
        **nucleus_kwargs,
    )

    label = make_3class_label_full(droplet_mask, nucleus_mask)

    metadata = {
        "focus_pass": True,
        "focus_score": focus_score,
        "droplet_area_px": int(droplet_mask.sum()),
        "nucleus_area_px": int(nucleus_mask.sum()),
        "reason": "included",
    }

    debug = {
        **droplet_debug,
        **nucleus_debug,
        "droplet_mask": droplet_mask,
        "nucleus_mask": nucleus_mask,
    }

    return label, metadata, debug


In [ ]:

def build_input_image(mem2d, nuc2d, npc2d):
    """
    Returns H×W×3 float32 in [0,1].

    Channel order:
        0 = nucleus channel
        1 = NPC channel
        2 = membrane/context channel
    """
    return np.stack(
        [
            normalize_channel(nuc2d),
            normalize_channel(npc2d),
            normalize_channel(mem2d),
        ],
        axis=-1,
    ).astype("float32")

In [ ]:
# ============================================================
# QC: Build and Visualize One Training Plane
# ============================================================

# Choose a representative in-focus Z plane.
t = 0
z = min(10, img_fov.shape[1] - 1)

npc2d = img_fov[t, z, :, :, NPC_CH]
nuc2d = img_fov[t, z, :, :, NUC_CH]

label_test, metadata_test, debug_test = build_training_label_plane(
    npc2d=npc2d,
    nuc2d=nuc2d,
    pixel_size_um=PIXEL_SIZE_UM,
    droplet_kwargs={
        "puncta_clip_quantile": 0.985,
        "blur_sigma": 8.0,
        "min_droplet_area_um2": 500,
        "hole_area_um2": 5000,
        "closing_radius_um": 3.0,
    },
    nucleus_kwargs={
        "min_nucleus_area_um2": 50,
        "blur_sigma": 1.5,
        "threshold_method": "otsu",
        "closing_radius_um": 1.5,
        "hole_area_um2": 200,
    },
)

print(metadata_test)

if label_test is not None:
    fig, ax = plt.subplots(2, 3, figsize=(15, 9))

    ax[0, 0].imshow(npc2d, cmap="gray")
    ax[0, 0].set_title("Raw NPC channel")

    ax[0, 1].imshow(debug_test["npc_blur"], cmap="gray")
    ax[0, 1].set_title("NPC clipped + low-pass blur")

    ax[0, 2].imshow(debug_test["droplet_mask"], cmap="gray")
    ax[0, 2].set_title("Droplet training mask")

    ax[1, 0].imshow(nuc2d, cmap="gray")
    ax[1, 0].set_title("Raw nucleus channel")

    ax[1, 1].imshow(debug_test["nucleus_mask"], cmap="gray")
    ax[1, 1].set_title("Filled nucleus training mask")

    ax[1, 2].imshow(label_test, cmap="viridis", vmin=0, vmax=2)
    ax[1, 2].set_title("Final label: 0 bg, 1 droplet, 2 nucleus")

    for axis in ax.ravel():
        axis.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Plane excluded:", metadata_test)


In [ ]:
# ============================================================
# QC: Focus Scores Across Z
# ============================================================


def compute_focus_scores_for_timepoint(img_fov, t=0, nuc_ch=NUC_CH):
    """Compute focus score for every Z plane at one timepoint."""
    _, n_z, _, _, _ = img_fov.shape
    rows = []

    for z in range(n_z):
        nuc2d = img_fov[t, z, :, :, nuc_ch]
        score = focus_score_variance_laplacian(nuc2d)
        rows.append({
            "t": t,
            "z": z,
            "focus_score": score,
            "edge_z_pass": z_plane_allowed(z, n_z),
            "focus_pass": score >= MIN_FOCUS_SCORE,
            "train_pass": z_plane_allowed(z, n_z) and score >= MIN_FOCUS_SCORE,
        })

    return pd.DataFrame(rows)


focus_df = compute_focus_scores_for_timepoint(img_fov, t=0)
display(focus_df)

plt.figure(figsize=(8, 4))
plt.plot(focus_df["z"], focus_df["focus_score"], marker="o")
plt.axhline(MIN_FOCUS_SCORE, linestyle="--")
plt.xlabel("Z plane")
plt.ylabel("Variance of Laplacian focus score")
plt.title("Focus score by Z plane")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Build U-Net Training Dataset from Memory-Mapped Hyperstack
# ============================================================


def build_input_image(mem2d, nuc2d, npc2d):
    """
    Returns H×W×3 float32 in [0,1].

    Channel order:
        0 = nucleus channel
        1 = NPC channel
        2 = membrane/context channel
    """
    return np.stack(
        [
            normalize_channel(nuc2d),
            normalize_channel(npc2d),
            normalize_channel(mem2d),
        ],
        axis=-1,
    ).astype("float32")


def build_unet_dataset_from_hyperstack(
    img_fov,
    out_images=IMG_OUT,
    out_labels=LAB_OUT,
    patch_size=PATCH_SIZE,
    stride=STRIDE,
    min_fg_frac=MIN_FG_FRAC,
    pixel_size_um=PIXEL_SIZE_UM,
    exclude_edge_z=EXCLUDE_EDGE_Z,
    min_focus_score=MIN_FOCUS_SCORE,
    use_focus_filter=USE_FOCUS_FILTER,
):
    """
    Build .npy image and label patches for U-Net training.

    This version:
    - reads planes lazily from tifffile.memmap
    - excludes edge Z planes
    - excludes low-focus Z planes
    - generates droplet labels from NPC low-frequency signal, not puncta
    - generates filled nuclear labels
    """
    n_t, n_z, height, width, n_c = img_fov.shape
    patch_id = 0
    plane_rows = []

    for t in tqdm(range(n_t), desc="Timepoints"):
        for z in range(n_z):
            if not z_plane_allowed(z, n_z, exclude_edge_z=exclude_edge_z):
                plane_rows.append({
                    "t": t,
                    "z": z,
                    "included": False,
                    "reason": "edge_z_excluded",
                    "focus_score": np.nan,
                    "patches_saved": 0,
                })
                continue

            mem2d = img_fov[t, z, :, :, MEM_CH]
            nuc2d = img_fov[t, z, :, :, NUC_CH]
            npc2d = img_fov[t, z, :, :, NPC_CH]

            label_3c, metadata, _ = build_training_label_plane(
                npc2d=npc2d,
                nuc2d=nuc2d,
                pixel_size_um=pixel_size_um,
                min_focus_score=min_focus_score,
                use_focus_filter=use_focus_filter,
            )

            if label_3c is None:
                plane_rows.append({
                    "t": t,
                    "z": z,
                    "included": False,
                    "reason": metadata["reason"],
                    "focus_score": metadata["focus_score"],
                    "patches_saved": 0,
                })
                continue

            input_image = build_input_image(mem2d, nuc2d, npc2d)
            patches_this_plane = 0

            for y in range(0, height - patch_size + 1, stride):
                for x0 in range(0, width - patch_size + 1, stride):
                    x_patch = input_image[y:y + patch_size, x0:x0 + patch_size, :]
                    y_patch = label_3c[y:y + patch_size, x0:x0 + patch_size]

                    fg_frac = np.mean(y_patch > 0)
                    if fg_frac < min_fg_frac:
                        continue

                    img_fname = out_images / f"img_t{t:03d}_z{z:03d}_y{y:04d}_x{x0:04d}_p{patch_id:06d}.npy"
                    lab_fname = out_labels / f"lab_t{t:03d}_z{z:03d}_y{y:04d}_x{x0:04d}_p{patch_id:06d}.npy"

                    np.save(img_fname, x_patch)
                    np.save(lab_fname, y_patch.astype("uint8"))

                    patch_id += 1
                    patches_this_plane += 1

            plane_rows.append({
                "t": t,
                "z": z,
                "included": True,
                "reason": metadata["reason"],
                "focus_score": metadata["focus_score"],
                "droplet_area_px": metadata["droplet_area_px"],
                "nucleus_area_px": metadata["nucleus_area_px"],
                "patches_saved": patches_this_plane,
            })

            del input_image, label_3c
            gc.collect()

    summary_df = pd.DataFrame(plane_rows)
    summary_path = QC_OUT / "training_plane_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    print(f"Finished. Saved {patch_id} patches.")
    print(f"Plane summary saved to: {summary_path}")
    return summary_df


In [ ]:
#### Use Command for building the Patches as defined above ####
build_unet_dataset_from_hyperstack(img_fov)


In [ ]:
#### DANGER!!!!!! CLEARS PREVIOUS PATCHES DON'T JUST PRESS THE BUTTON YOU MONKEY!#### 
#import shutil
#shutil.rmtree(IMG_OUT)
#shutil.rmtree(LAB_OUT)
#IMG_OUT.mkdir(parents=True, exist_ok=True)
#LAB_OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
#### Sanity Check to List Saved Image and Label Patches ####
list(IMG_OUT.iterdir()), list(LAB_OUT.iterdir())


In [ ]:
# ============================================================
# Saved Patch Count
# ============================================================

print("Images directory:", IMG_OUT)
print("Labels directory:", LAB_OUT)
print("Number of images:", len(list(IMG_OUT.glob("*.npy"))))
print("Number of labels:", len(list(LAB_OUT.glob("*.npy"))))


In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt

# pick a random image patch
example_img = random.choice(list(IMG_OUT.glob("*.npy")))

# convert its filename to the matching label filename
example_lab = LAB_OUT / example_img.name.replace("img_", "lab_")

# load
x = np.load(example_img)
y = np.load(example_lab)

print("Patch image shape:", x.shape)
print("Patch label shape:", y.shape)

# visualize
fig, ax = plt.subplots(1, 4, figsize=(14, 4))
ax[0].imshow(x[..., 0], cmap='gray'); ax[0].set_title("Nuclear-import"); ax[0].axis("off")
ax[1].imshow(x[..., 1], cmap='gray'); ax[1].set_title("NPC");            ax[1].axis("off")
ax[2].imshow(x[..., 2], cmap='gray'); ax[2].set_title("Membrane");       ax[2].axis("off")
ax[3].imshow(y, cmap='tab20');         ax[3].set_title("Label (0/1/2)"); ax[3].axis("off")
plt.tight_layout(); plt.show()



U-Net Machine Learning Approach

In [ ]:
import tensorflow as tf
print("TF version:", tf.__version__)


In [ ]:
from pathlib import Path
import numpy as np

# Root where your patches live (adjust if needed)
OUT_ROOT = Path(r"C:/Users/cowboy/OneDrive/Documents/Unviversity of Alabama/Nuclear_Scaling/ML_training")
IMG_OUT  = OUT_ROOT / "images"
LAB_OUT  = OUT_ROOT / "labels"

# Basic settings
PATCH_SIZE   = 256          # must match your patch builder
NUM_CHANNELS = 3            # [nuc import, NPC, membrane]
NUM_CLASSES  = 3            # background, droplet, nucleus
BATCH_SIZE   = 8            # tune based on GPU memory
VAL_FRAC     = 0.1          # fraction of patches for validation

# List patch files (will be reused after you rebuild)
img_paths = sorted(IMG_OUT.glob("*.npy"))
lab_paths = sorted(LAB_OUT.glob("*.npy"))

print(f"Found {len(img_paths)} image patches, {len(lab_paths)} label patches.")


In [ ]:
# Build dictionaries keyed by the shared ID part of the filename
img_by_id = {}
for p in img_paths:
    patch_id = p.name.replace("img_", "")  # everything after 'img_'
    img_by_id[patch_id] = p

lab_by_id = {}
for p in lab_paths:
    patch_id = p.name.replace("lab_", "")  # everything after 'lab_'
    lab_by_id[patch_id] = p

# Intersection of IDs that have BOTH image and label
common_ids = sorted(set(img_by_id.keys()) & set(lab_by_id.keys()))
print("Common patches:", len(common_ids))

# Now build *paired* lists
paired_img_paths = [img_by_id[i] for i in common_ids]
paired_lab_paths = [lab_by_id[i] for i in common_ids]

print("Paired images:", len(paired_img_paths))
print("Paired labels:", len(paired_lab_paths))


In [ ]:
import numpy as np

VAL_FRAC = 0.1  # or whatever you like

n = len(paired_img_paths)
indices = np.arange(n)
np.random.shuffle(indices)

n_val = int(n * VAL_FRAC)
val_idx   = indices[:n_val]
train_idx = indices[n_val:]

train_img_paths = [paired_img_paths[i] for i in train_idx]
train_lab_paths = [paired_lab_paths[i] for i in train_idx]
val_img_paths   = [paired_img_paths[i] for i in val_idx]
val_lab_paths   = [paired_lab_paths[i] for i in val_idx]

print(f"Train patches: {len(train_img_paths)} | Val patches: {len(val_img_paths)}")


In [ ]:
import numpy as np
import tensorflow as tf

def load_npy_pair(img_path, lab_path):
    """
    Python function used by tf.numpy_function.

    img_path, lab_path come in as 0-d numpy arrays of dtype=bytes,
    or sometimes as np.bytes_. We convert them to normal Python strings,
    then load with np.load.
    """
    # Handle 0-d numpy arrays
    if isinstance(img_path, np.ndarray):
        img_path = img_path.item()
    if isinstance(lab_path, np.ndarray):
        lab_path = lab_path.item()

    # Now img_path / lab_path should be bytes; decode to str
    if isinstance(img_path, bytes):
        img_path = img_path.decode("utf-8")
    if isinstance(lab_path, bytes):
        lab_path = lab_path.decode("utf-8")

    img = np.load(img_path)   # (H,W,3)
    lab = np.load(lab_path)   # (H,W)

    return img.astype("float32"), lab.astype("int32")


def tf_load_npy_pair(img_path, lab_path):
    img, lab = tf.numpy_function(
        load_npy_pair,
        [img_path, lab_path],
        [tf.float32, tf.int32]
    )

    # Set static shapes so TF knows dimensions
    img.set_shape((PATCH_SIZE, PATCH_SIZE, NUM_CHANNELS))
    lab.set_shape((PATCH_SIZE, PATCH_SIZE))

    # One-hot encode labels to (H, W, NUM_CLASSES)
    lab_oh = tf.one_hot(lab, depth=NUM_CLASSES, dtype=tf.float32)
    return img, lab_oh


In [ ]:
train_ds = make_dataset(train_img_paths, train_lab_paths, BATCH_SIZE, shuffle=True)
val_ds   = make_dataset(val_img_paths,   val_lab_paths,   BATCH_SIZE, shuffle=False)


In [ ]:
for x_batch, y_batch in train_ds.take(1):
    print("x_batch:", x_batch.shape, x_batch.dtype)
    print("y_batch:", y_batch.shape, y_batch.dtype)


In [ ]:
#### Build U-Net Model for 3-Class Segmentation ####
from tensorflow.keras import layers, models

def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x

def encoder_block(x, filters):
    c = conv_block(x, filters)
    p = layers.MaxPooling2D((2, 2))(c)
    return c, p

def decoder_block(x, skip, filters):
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding="same")(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x

def build_unet(input_shape=(PATCH_SIZE, PATCH_SIZE, NUM_CHANNELS),
               num_classes=NUM_CLASSES):
    inputs = layers.Input(shape=input_shape)

    # Encoder
    c1, p1 = encoder_block(inputs, 64)
    c2, p2 = encoder_block(p1, 128)
    c3, p3 = encoder_block(p2, 256)
    c4, p4 = encoder_block(p3, 512)

    # Bottleneck
    bn = conv_block(p4, 1024)

    # Decoder
    d1 = decoder_block(bn, c4, 512)
    d2 = decoder_block(d1, c3, 256)
    d3 = decoder_block(d2, c2, 128)
    d4 = decoder_block(d3, c1, 64)

    # Output: per-pixel softmax over 3 classes
    outputs = layers.Conv2D(num_classes, 1, activation="softmax")(d4)

    model = models.Model(inputs, outputs, name="UNet_droplet_nucleus")
    return model

model = build_unet()
model.summary()


Compile and Train U-Net Model 


In [ ]:
#### Compile Model with Custom IoU Metric ####
loss = tf.keras.losses.CategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)

def iou_metric(y_true, y_pred, eps=1e-6):
    # y_true, y_pred: (B,H,W,C) one-hot
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # predicted class via argmax
    y_pred_cls = tf.argmax(y_pred, axis=-1)
    y_true_cls = tf.argmax(y_true, axis=-1)

    # ignore background (class 0) if you want; here we compute macro IoU
    ious = []
    for c in range(1, NUM_CLASSES):  # skip 0 if you like
        y_true_c = tf.cast(tf.equal(y_true_cls, c), tf.float32)
        y_pred_c = tf.cast(tf.equal(y_pred_cls, c), tf.float32)
        inter = tf.reduce_sum(y_true_c * y_pred_c)
        union = tf.reduce_sum(y_true_c) + tf.reduce_sum(y_pred_c) - inter + eps
        ious.append((inter + eps) / union)
    return tf.reduce_mean(ious)

model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=["accuracy", iou_metric]
)


In [ ]:
# ============================================================
# Train Model with Modern Keras Saving Format
# ============================================================

EPOCHS = 5

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_delta=1e-4,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

model.save(FINAL_MODEL_PATH)
print("Saved final model to:", FINAL_MODEL_PATH)


In [ ]:
#### Visualize Example Predictions on Validation Set ####
import matplotlib.pyplot as plt

# take one batch from val set
for x_batch, y_batch in val_ds.take(1):
    break

y_pred = model.predict(x_batch)

x0 = x_batch[0]
y0_true = y_batch[0]
y0_pred = y_pred[0]

y0_true_cls = tf.argmax(y0_true, axis=-1).numpy()
y0_pred_cls = tf.argmax(y0_pred, axis=-1).numpy()

fig, ax = plt.subplots(1, 5, figsize=(16, 4))
ax[0].imshow(x0[..., 0], cmap="gray"); ax[0].set_title("Nuclear-import"); ax[0].axis("off")
ax[1].imshow(x0[..., 1], cmap="gray"); ax[1].set_title("NPC");            ax[1].axis("off")
ax[2].imshow(x0[..., 2], cmap="gray"); ax[2].set_title("Membrane");       ax[2].axis("off")
ax[3].imshow(y0_true_cls, cmap="tab20"); ax[3].set_title("GT label");     ax[3].axis("off")
ax[4].imshow(y0_pred_cls, cmap="tab20"); ax[4].set_title("Pred label");   ax[4].axis("off")
plt.tight_layout(); plt.show()
